# Debate results analysis (PRD §9)

This notebook presents the four key analyses of the committed debate runs:

1. **Who-wins distribution** — how often each side (pro / con / tie) won.
2. **Agree-vs-disagree rate** — how often the agents converged.
3. **Drift / nudge frequency per side** — evidence the anti-sycophancy design works.
4. **Tokens & latency per round and per topic**.

All reusable logic lives in the **tested** module `agent_debate.core.research`
(aggregation from task 14.1, analysis functions from task 14.2). This notebook is
deliberately thin: it imports the module, calls pure functions, and shows the
results as tables. Rich charts are the separate task 14.3 — `matplotlib`/`pandas`
are not workspace dependencies, so we keep 14.2 to clear tables/numbers and leave
visualisation to 14.3.

Run it with:

```bash
uv run --with jupyter jupyter nbconvert --to notebook --execute notebooks/analysis.ipynb
```

In [ ]:
from pathlib import Path

from agent_debate.core.research import (
    aggregate_runs,
    agree_vs_disagree,
    nudges_per_side,
    round_metrics,
    tokens_latency_per_topic,
    who_wins,
)
from agent_debate.log import DEFAULT_RUNS_DIR


# No hard-coded path: the runs dir name comes from the LOG package default
# (task 14.1). Anchor it to the repo root (the first parent holding the workspace
# pyproject.toml) so the notebook works regardless of the kernel's working
# directory and never picks up a stray LOG-chatter "runs" folder elsewhere.
def _runs_dir() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "pyproject.toml").is_file() and (base / DEFAULT_RUNS_DIR).is_dir():
            return base / DEFAULT_RUNS_DIR
    return Path(DEFAULT_RUNS_DIR)


RUNS_DIR = _runs_dir()
summaries = aggregate_runs(RUNS_DIR)
print(f"Loaded {len(summaries)} debate run(s) from {RUNS_DIR}/")
for s in summaries:
    print(f"  {s.run_id}: winner={s.winner} rounds={s.rounds} tokens={s.total_tokens}")

## 1. Who-wins distribution

Counts of runs won by each side across all topics (ties included). A healthy
experiment shows the verdict is not biased toward a single side.

In [ ]:
dist = who_wins(summaries)
print(f"{'winner':<8} {'runs':>5}")
for side, count in sorted(dist.items()):
    print(f"{side:<8} {count:>5}")

## 2. Agree-vs-disagree rate

Whether the controller judged the agents to have **converged** (agreed) or stayed
opposed. The debate format is adversarial, so we expect mostly *disagree*.

In [ ]:
rates = agree_vs_disagree(summaries)
print(f"agree    (converged):     {rates['agree']}")
print(f"disagree (not converged): {rates['disagree']}")
print(f"agree rate:               {rates['agree_rate']:.0%}")

## 3. Drift / nudge frequency per side (anti-sycophancy evidence)

How often the controller had to **nudge** a captured debater back onto its
assigned side. A low or zero count is direct evidence that the separate-context,
side-anchored relay design keeps either agent from controlling the other.

> Note: the committed sample runs needed **no** nudges (0/0). The staged-drift
> detection path is exercised by the engine tests (task 8.4); the real debates
> simply never drifted enough to require a correction.

In [ ]:
nudges = nudges_per_side(summaries)
print(f"pro nudges (total):  {nudges.pro_total}")
print(f"con nudges (total):  {nudges.con_total}")
print(f"all nudges (total):  {nudges.total}")
print()
print(f"{'run_id':<28} {'pro':>4} {'con':>4}")
for run_id, (pro, con) in sorted(nudges.per_run.items()):
    print(f"{run_id:<28} {pro:>4} {con:>4}")

## 4. Tokens & latency

### 4a. Per topic (run-level)

Total tokens and average per-message latency for each topic/run.

In [ ]:
topics = tokens_latency_per_topic(summaries)
print(f"{'run_id':<28} {'rounds':>6} {'tokens':>8} {'avg latency ms':>15}")
for t in topics:
    print(f"{t.run_id:<28} {t.rounds:>6} {t.total_tokens:>8} {t.avg_latency_ms:>15.1f}")

### 4b. Per round (round-level)

Round-level tokens and latency are not in the run summary, so we parse them
straight from each run's JSONL event log via `round_metrics`. Below we show one
representative run; change `RUN_ID` to inspect another.

In [ ]:
RUN_ID = summaries[0].run_id if summaries else ""
lines = (RUNS_DIR / f"{RUN_ID}.jsonl").read_text(encoding="utf-8").splitlines()
rounds = round_metrics(lines)
print(f"Round-level metrics for run: {RUN_ID}\n")
header = f"{'round':>5} {'tokens':>8} {'latency ms':>11} {'pro tok':>8} {'con tok':>8}"
print(header)
for r in rounds:
    print(f"{r.round:>5} {r.tokens:>8} {r.latency_ms:>11.0f} {r.pro_tokens:>8} {r.con_tokens:>8}")

## Summary

- **Who-wins** is spread across pro / con / tie — the verdict is not one-sided.
- **Agree-vs-disagree**: the agents stayed opposed in every committed run
  (0% agree), as expected for an adversarial format.
- **Nudges**: zero on both sides — the anti-sycophancy design held without the
  controller having to intervene.
- **Tokens/latency** grow with round count and message length; the per-round
  table shows where the cost accumulates within a single debate.

Charts that visualise these tables are added in task 14.3.